<a href="https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines the data contract for the Search Intelligence ranking task.

The contract specifies:
- the unit of analysis,
- the development time window,
- allowed features,
- the future label,
- contextual fields,
- excluded/leaky fields,
- verification checks,
- and known data limitations.

## 1. Unit of analysis + time window
### Unit of analysis
The raw `fact_content_daily_performance` table is at daily grain:
**one row = one content item × one client × one report date.**
The identifiers `client_hash_id` and `content_hash_id` identify the client and content item, while `report_date` identifies the daily observation.
For the eventual ranking task, daily observations can be aggregated into a page-level feature record at a defined decision moment.
### Development window
I use **March 2026** as the development month.
March is a middle-panel month rather than the final month, which helps avoid using the final period during development.
### Decision point
Features must be calculated only from information available at or before the decision moment.
The future target window must occur after the feature window. Information from the target period must not be used as a feature.

### Verification Query 1 — Grain

I verify that the daily performance table has one observation per combination of report date, client, and content. I compare the total number of rows with the number of distinct `(report_date, client_hash_id, content_hash_id)` combinations.

In [36]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS distinct_grain_rows,
        COUNT(*) -
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────┐
│ total_rows │ distinct_grain_rows │ duplicate_rows │
│   int64    │        int64        │     int64      │
├────────────┼─────────────────────┼────────────────┤
│    9841378 │             9841378 │              0 │
└────────────┴─────────────────────┴────────────────┘

### Interpretation

The total row count matches the number of distinct date × client × content combinations, confirming that the selected March slice follows the expected daily grain.

### Verification Query 2 — Row count and date span

I verify the actual number of observations and the minimum and maximum report dates represented in the March 2026 development slice.

In [37]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

date_check

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘

### Interpretation

The query confirms the actual size and date coverage of the March 2026 development slice instead of assuming that the partition name alone describes the available observations.

### Verification Query 3 — GA4 availability

GA4 availability is checked explicitly using `IS TRUE`. This ensures that only confirmed TRUE values are counted as available and that NULL values are not silently interpreted as available.

In [39]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

availability_check

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘

### Interpretation

GA4 availability is not assumed for every observation. Only rows where `ga4_data_available IS TRUE` are treated as confirmed GA4-available observations.

## 2. Fields: feature / label / context / excluded

### Features

The initial feature frame is limited to five observable signals:

| Feature | Why it is useful | Available when? |
|---|---|---|
| `gsc_impressions` | Measures how often the content received search impressions | Knowable at the decision moment because it is calculated from the completed feature window |
| `gsc_clicks` | Measures search clicks received by the content | Knowable at the decision moment because it is observed before the future target window |
| `gsc_sum_position` | Measures the aggregate search-position signal | Knowable at the decision moment because it summarizes observations from the feature window |
| `sessions_organic` | Measures organic sessions reaching the content | Knowable at the decision moment when the GA4 observation is available in the feature window |
| `scroll_events` | Provides an engagement signal from observed page interaction | Knowable at the decision moment when GA4 data is available in the feature window |

All five features must be constructed only from observations available before the decision moment. No target-window observations are allowed in the feature values.


### Label / target

The eventual target represents a future search-performance outcome used to prioritize content for review.

The target must be calculated from a future window after the feature/decision window. This prevents current or future outcome information from being included in the feature set.

The current observed GSC metrics are therefore treated as historical signals rather than automatically being used as the future target.


### Context

The following fields are retained for identification, grouping, joining, and time-window construction:

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

These fields are contextual metadata and are not predictive features.

### Excluded fields

Identifiers such as `client_hash_id` and `content_hash_id` are excluded from model features because they identify entities rather than represent generalizable measurements.

`report_date` and `month` are used for constructing and validating time windows rather than as predictive features.

Label-derived or outcome-derived fields are excluded because they can leak information about the target.

Availability flags such as `gsc_data_available` and `ga4_data_available` are used to determine whether measurements are valid/available rather than being treated automatically as predictive signals.

Any feature calculated using observations from the future target window is excluded.

In particular, `trend_direction` and `trend_pct`, if present in a derived dataset, must not be used as model features because they can directly reveal the outcome.

## 3. Verify it with queries

The data contract is verified using three small queries on the March 2026 development slice.

1. **Grain:** verify that each row represents one report date × client × content combination.
2. **Row count + date span:** verify the number of rows and the actual date range.
3. **Availability:** verify GA4 availability using `IS TRUE`.

These checks are performed on the March 2026 middle-panel month rather than the final month.

### Query 1 — Grain check

I verify that the raw daily performance table has one observation for each combination of `report_date`, `client_hash_id`, and `content_hash_id`.

The total number of rows is compared with the number of distinct date × client × content combinations. A zero difference indicates that there are no duplicate observations at the expected grain.

In [40]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS distinct_grain_rows,
        COUNT(*) -
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────┐
│ total_rows │ distinct_grain_rows │ duplicate_rows │
│   int64    │        int64        │     int64      │
├────────────┼─────────────────────┼────────────────┤
│    9841378 │             9841378 │              0 │
└────────────┴─────────────────────┴────────────────┘

### Interpretation

The March 2026 slice contains 9,841,378 rows and 9,841,378 distinct report date × client × content combinations. The duplicate count is 0, confirming that the expected daily grain is unique for this slice.

Therefore, one row represents one content item for one client on one report date.



The result confirms that the March 2026 slice follows the expected daily grain, with no duplicate date × client × content combinations.

### Query 2 — Row count and date span

I verify the number of observations in the March 2026 development slice and inspect the minimum and maximum `report_date`.

This confirms the actual temporal coverage of the selected partition.

In [41]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

date_check

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘

### Interpretation

The March 2026 development slice contains 9,841,378 observations covering 2026-03-01 through 2026-03-31. The date range confirms that the selected partition contains the complete March calendar month.

### Query 3 — GA4 availability

I verify GA4 availability explicitly with `IS TRUE`.

This is important because availability fields can contain TRUE, FALSE, or NULL. Only an explicit TRUE value is treated as confirmed availability.

In [42]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

availability_check

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘

### Interpretation

Of the 9,841,378 March 2026 observations, 413,966 have confirmed GA4 availability using `ga4_data_available IS TRUE`, which is approximately 4.21% of the rows.

This demonstrates that GA4 coverage is substantially more limited than the total warehouse coverage. GA4-derived features therefore require explicit availability handling and should not treat unavailable observations as zero engagement.

## 4. Data limits

### Data limitations

#### 1. Unbalanced history

The warehouse is an unbalanced panel. Different clients can have different amounts of historical data, so the same calendar period does not necessarily provide identical historical coverage for every client.

Client-specific data-start dates should therefore be checked before constructing longer feature and target windows.

#### 2. GSC-only / incomplete analytics history

Some observations may have search data available while GA4 data is unavailable. Therefore, missing GA4 measurements should not automatically be interpreted as zero engagement.

Availability flags must be checked explicitly before using GA4-derived features.

#### 3. Feature/label window overlap

Feature and target windows must remain separate. If a feature is calculated using observations from the future target period, it leaks information about the outcome.

The final target period must therefore be kept out of the feature construction process.

#### 4. Final-month contamination

The final month should be treated as a held-out period rather than being used during development. A middle month such as March 2026 is used for development and contract verification.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.